In [2]:
import cv2
import os
import urllib.request

prototxt_url = "https://github.com/opencv/opencv/raw/master/samples/dnn/face_detector/deploy.prototxt"
model_url = "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"

prototxt_file = "deploy.prototxt"
model_file = "res10_300x300_ssd_iter_140000.caffemodel"

# def download_if_missing(url, filename):
#     if not os.path.exists(filename):
#         print(f"Downloading {filename} ...")
#         urllib.request.urlretrieve(url, filename)
#         print(f"{filename} downloaded.")
#     else:
#         print(f"{filename} already exists.")


# download_if_missing(prototxt_url, prototxt_file)
# download_if_missing(model_url, model_file)

# Load DNN model
net = cv2.dnn.readNetFromCaffe(prototxt_file, model_file)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)), 1.0,
                                 (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > 0.5:  # detection threshold
            box = detections[0, 0, i, 3:7] * [w, h, w, h]
            (x1, y1, x2, y2) = box.astype("int")

            # Ensure box is within frame
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w-1, x2), min(h-1, y2)

            # Blur face region
            roi = frame[y1:y2, x1:x2]
            if roi.size > 0:
                roi_blur = cv2.GaussianBlur(roi, (99, 99), 30)
                frame[y1:y2, x1:x2] = roi_blur

    cv2.imshow("Real-time Face Blur (DNN)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 